# Output Parser (구조화된 출력 파싱) 실습

LLM의 답변을 자유 텍스트가 아니라 **미리 정한 구조(스키마)를 가진 파이썬 객체**로 바로 받아오는 방법을 실습했다. `PydanticOutputParser`로 직접 파싱 과정을 구성해보고, 마지막에는 `with_structured_output()`으로 더 간단하게 같은 결과를 얻는 방법까지 비교한다.

## 1. 환경변수 로드

`.env`에 저장된 `OPENAI_API_KEY`를 불러온다.

In [1]:
from dotenv import load_dotenv

# .env 파일의 OPENAI_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. 사용할 도구 임포트 및 LLM 준비

- `PydanticOutputParser` : LLM이 낸 텍스트(JSON 형태)를 Pydantic 모델 객체로 변환해주는 파서.
- `BaseModel`, `Field` : Pydantic으로 "내가 원하는 답변 구조(스키마)"를 정의할 때 사용.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# temperature=0 : 답변을 최대한 일관되게 받기 위해 무작위성을 낮춘다.
llm = ChatOpenAI(
    temperature=0,
    model="gpt-4.1-mini"
)

## 3. 예시로 사용할 이메일 본문 준비

구조화해서 뽑아낼 대상 텍스트. 발신자, 수신자, 제목, 요청 사항, 미팅 일정 등이 섞여 있는 평범한 비즈니스 이메일이다. 이 안에서 필요한 정보만 골라내는 것이 목표다.

In [3]:
# 구조화해서 정보를 추출할 대상: 미팅을 제안하는 비즈니스 이메일 본문.
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

## 4. Output Parser 없이 그냥 요약해보기 (자유 텍스트)

아직 구조화 파서를 쓰지 않고, "중요한 내용을 추출해달라"는 프롬프트만으로 먼저 호출해본다. 결과는 사람이 읽기엔 좋지만 형식이 자유로운 텍스트라서, 프로그램이 특정 필드(날짜, 이메일 주소 등)만 꺼내 쓰기는 어렵다. 이 문제를 다음 단계에서 `PydanticOutputParser`로 해결한다.

> 참고: `from itertools import chain`은 실습 코드에 남아있던 import인데, 바로 아래에서 `chain = prompt | llm`으로 같은 이름의 변수를 덮어써서 실제로는 itertools의 chain은 사용되지 않는다.

In [5]:
from itertools import chain
from langchain_core.prompts import PromptTemplate

# 이메일 본문을 넣고 "중요한 내용을 추출해달라"고만 요청하는 단순 프롬프트.
prompt = PromptTemplate.from_template(
    template="다음의 이메일 내용중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)
llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")

# chain 변수명이 위 import의 itertools.chain을 덮어쓴다(이후로는 이 체인 객체를 가리킴).
chain = prompt | llm
answer = chain.stream({"email_conversation": email_conversation})
for chunk in answer:
    print(chunk.content, end="", flush=True)

다음은 이메일의 중요한 내용입니다:

1. 발신자: 김철수 상무 (바이크코퍼레이션)
2. 수신자: 이은채 대리 (테디인터내셔널)
3. 요청 사항:
   - "ZENESIS" 자전거에 대한 상세 브로슈어 요청
   - 특히 기술 사양, 배터리 성능, 디자인 정보 필요
4. 목적: 유통 전략 및 마케팅 계획 구체화를 위한 정보 확보
5. 미팅 제안:
   - 일시: 1월 15일 화요일 오전 10시
   - 장소: 귀사 사무실
   - 내용: 협력 가능성 논의

## 5. 원하는 출력 구조를 Pydantic 모델로 정의

이메일에서 뽑아내고 싶은 필드를 `BaseModel`로 정의한다. 각 필드의 `description`은 LLM에게 "이 필드에는 어떤 내용을 넣어야 하는지" 알려주는 설명으로 쓰인다.

- `person` : 보낸 사람
- `email` : 보낸 사람 이메일 주소
- `subject` : 메일 제목
- `summary` : 본문 요약
- `date` : 언급된 미팅 날짜/시간

이 스키마를 `PydanticOutputParser`에 넘기면, "이 형식대로 답해달라"는 지시문 생성과 "실제로 온 답을 이 형식으로 파싱"하는 것을 모두 담당해준다.

In [7]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

# EmailSummary 스키마를 기준으로 LLM 출력을 파싱해줄 파서.
parser = PydanticOutputParser(pydantic_object=EmailSummary)

## 6. 파서가 요구하는 출력 형식(포맷 지시문) 확인

`parser.get_format_instructions()`는 "LLM이 어떤 형태(JSON 스키마)로 답해야 하는지"를 설명하는 텍스트를 만들어준다. 이 텍스트를 프롬프트에 그대로 끼워 넣으면, LLM이 그 형식에 맞춰 JSON을 출력하도록 유도할 수 있다.

In [8]:
# EmailSummary 스키마를 설명하는 JSON 포맷 지시문을 출력해본다.
# 이 문자열을 프롬프트에 넣어서 LLM에게 "이 형식으로 답해줘"라고 알려준다.
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


## 7. 포맷 지시문을 반영한 프롬프트 작성

`{format}` 자리에 방금 확인한 포맷 지시문을 `partial()`로 미리 고정해 둔다. 이렇게 하면 실제로 체인을 호출할 때는 `question`, `email_conversation`만 채우면 되고, `format`은 매번 다시 넣지 않아도 된다.

In [10]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# format 변수를 parser의 포맷 지시문으로 미리 고정(partial)해 둔다.
# -> 이후 호출 시에는 question, email_conversation만 채우면 된다.
prompt = prompt.partial(format=parser.get_format_instructions())

## 8. 아직 parser는 연결하지 않은 체인

일부러 `prompt | llm`까지만 연결해서, LLM이 실제로 어떤 원본 텍스트(JSON 문자열)를 출력하는지 먼저 확인해본다.

In [11]:
# 아직 parser를 붙이지 않은 체인. LLM이 반환하는 원본 텍스트를 그대로 받는다.
chain = prompt | llm

## 9. 체인 호출 및 스트리밍 출력 누적

스트리밍으로 답을 받으면서, 화면에 출력함과 동시에 `output` 문자열에 이어붙여 전체 응답을 모아둔다. 이렇게 모은 `output`을 다음 단계에서 파서에 넘겨 구조화된 객체로 변환한다.

출력 결과를 보면 LLM이 마크다운 코드블록(```json ... ```)에 감싸인 JSON 텍스트를 응답하는 것을 볼 수 있다.

In [14]:
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해주세요."
    }
)

output = ""

# 스트리밍으로 오는 chunk를 화면에 출력하면서, 동시에 output 문자열에 누적한다.
# (뒤에서 parser.parse(output)로 이 전체 텍스트를 구조화된 객체로 변환할 것이다.)
for chunk in response:
    text = chunk.content
    if isinstance(text, str):
        print(text, end="", flush=True)
        output += text

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 ZENESIS 자전거에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하고, 유통 전략과 마케팅 계획 협의를 위해 1월 15일 화요일 오전 10시에 미팅을 제안함.",
  "date": "1월 15일 화요일 오전 10시"
}
```

## 10. 모아둔 텍스트를 파서로 직접 파싱

`parser.parse(output)`을 호출하면, LLM이 낸 JSON 텍스트(```json ... ``` 코드블록 포함)를 알아서 파싱해서 `EmailSummary` 객체로 변환해준다. 이제 `structured_output.person`, `structured_output.date`처럼 필드 단위로 값을 꺼내 쓸 수 있다.

In [16]:
# 앞에서 모아둔 JSON 텍스트(output)를 EmailSummary 객체로 파싱한다.
# 이제부터는 문자열이 아니라 구조화된 객체(필드 접근 가능)로 다룰 수 있다.
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하고, 유통 전략과 마케팅 계획 협의를 위해 1월 15일 화요일 오전 10시에 미팅을 제안함.' date='1월 15일 화요일 오전 10시'


## 11. parser까지 체인에 직접 연결하기

매번 `chain.stream()` → 텍스트 누적 → `parser.parse()`를 따로 호출하는 대신, 체인 자체에 `| parser`를 붙이면 LLM 응답을 자동으로 파싱까지 마친 `EmailSummary` 객체로 바로 받을 수 있다.

In [22]:
# prompt -> llm -> parser 순서로 연결.
# 이제 체인을 호출하면 파싱까지 자동으로 끝난 EmailSummary 객체가 바로 나온다.
chain = prompt | llm | parser

## 12. parser가 연결된 체인 호출

스트리밍 대신 `invoke()`로 한 번에 호출한다. 결과가 곧바로 `EmailSummary` 객체로 나오는 것을 확인할 수 있다 (문자열을 다시 파싱하는 과정이 필요 없다).

In [24]:
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해주세요."
    }
)

# chain에 parser가 포함되어 있으므로, response가 곧바로 EmailSummary 객체다.
print(response)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 ZENESIS 자전거의 상세 브로슈어(기술 사양, 배터리 성능, 디자인) 요청과 유통 및 마케팅 협력 가능성 논의를 위해 미팅을 제안함.' date='1월 15일 오전 10시'


## 13. 더 간단한 방법: with_structured_output()

`PromptTemplate` + `PydanticOutputParser`를 직접 조합하지 않고, `ChatOpenAI(...).with_structured_output(EmailSummary)`를 쓰면 모델이 함수 호출(도구 호출) 방식으로 곧바로 정해진 스키마의 객체를 반환하도록 LangChain이 대신 처리해준다. 포맷 지시문을 프롬프트에 직접 넣을 필요도 없다.

In [25]:
# with_structured_output(EmailSummary) : 프롬프트에 포맷 지시문을 직접 넣지 않아도
# 모델이 EmailSummary 스키마에 맞는 구조화된 객체를 반환하도록 만들어준다.
llm_with_structured = ChatOpenAI(
    temperature=0,
    model="gpt-4.1-mini"
).with_structured_output(EmailSummary)

## 14. 이메일 본문만 바로 넘겨서 호출

프롬프트 템플릿 없이 이메일 본문 문자열을 그대로 `invoke()`에 넘겨도 `EmailSummary` 객체가 바로 나온다. 다만 `date` 필드처럼 원문에 없는 정보(예: 실제 연도)는 모델이 임의로 채워 넣을 수 있으므로("2024-01-08"처럼), 구조화 출력이라고 해서 내용까지 항상 정확한 것은 아니라는 점을 확인할 수 있었다.

In [28]:
answer = llm_with_structured.invoke(email_conversation)
print(answer)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='김철수 상무는 이은채 대리에게 바이크코퍼레이션이 자전거 제조 및 유통 분야에서의 경험을 바탕으로 귀사의 신규 자전거 "ZENESIS"에 대한 상세 브로슈어(기술 사양, 배터리 성능, 디자인)를 요청하며, 협력 가능성 논의를 위해 1월 15일 화요일 오전 10시에 미팅을 제안합니다.' date='2024-01-08'
